In [ ]:
"""
<RAG(Retrieval Augmented Generation)>
 - RAG를 하는 이유 : 할루시네이션(환각 현상)을 줄이기 위해서 함
 - "검색 증강 생성"이라고 칭함
 - RAG = 대규모 언어 모델과
          + 외부 데이터(문서파일, 웹데이터, 벡터 데이터베이스)의 정보에서
            검색한 유사 정보를 결합하여
       = 정확하고 신뢰성 높은 결과를 생성하는 기술


 - 외부 데이터베이스 생성 방법
   1. 요청(질문) 주제에 맞는 -> 문서 로더
   2. 로드된 문서를 벡터 데이터베이스에 저장
      - 벡터 데이터베이스(행과 열의 묶음이 여러개 있는 데이터를 벡터 데이터 베이스 라고 함)
        : 크로마(Chroma) 또는 파이스(FAISS) 외에 다양한 벡터 DB가 있음


<문서 로더(Document Loader)>
 * 랭체인의 문서 로더는 다양한 데이터 소스와 형식을
   - 랭체인 프레임워크에서 사용할 수 있도록
   - 표준화된 문서 형식으로 변환하는 도구임


 * 80개 이상의 다양한 유형을 지원함
   - 텍스트, PDF, 웹페이지, CSV 등 다양한 형식의 데이터를 처리할 수 있음


 * 모든 로더는 데이터를 content와 metadata를 포함하는
   - 표준 문서 객체로 변환하여 랭체인의 다른 컴포넌트와 쉽게 통합할 수 있음


<문서 로더의 클래스(class) 종류>
 * WebBaseLoader
   - 웹 페이지(HTML 데이터)의 내용을 로드하는 기능을 제공함
 * PyPDFLoader
   - PDF 파일을 로드하고 필요에 따라 페이지 단위로 분할하는 기능을 제공함
 * CSVLoader
   - CSV 파일을 로드하는 기능을 제공함
 * DirectoryLoader
   - 지정된 디렉토리 내의 모든 파일을 로드하는 기능을 제공함
 * UnstructuredLoader
   - 다양한 형식의 파일을 로드하는 기능을 제공함


<문서 로더 라이브러리 설치>
  * Web, PDF, CSV 등등 파일 및 폴더 로드 처리 (이미 설치되어 있음)
    >uv pip install langchain_community


  * 로드된 웹 페이지 데이터 처리 라이브러리(이미 설치된 경우 안해도 됨)
    >uv pip install beautifulsoup4
"""


'\n<RAG(Retrieval Augmented Generation)>\n - "검색 증강 생성"이라고 칭함\n - RAG = 대규모 언어 모델과\n          + 외부 데이터(문서파일, 웹데이터, 벡터 데이터베이스)의 정보에서\n            검색한 유사 정보를 결합하여\n       = 정확하고 신뢰성 높은 결과를 생성하는 기술\n\n\n - 외부 데이터베이스 생성 방법\n   1. 요청(질문) 주제에 맞는 -> 문서 로더\n   2. 로드된 문서를 벡터 데이터베이스에 저장\n      - 벡터 데이터베이스(행과 열의 묶음이 여러개 있는 데이터를 벡터 데이터 베이스 라고 함)\n        : 크로마(Chroma) 또는 파이스(FAISS) 외에 다양한 벡터 DB가 있음\n\n\n<문서 로더(Document Loader)>\n * 랭체인의 문서 로더는 다양한 데이터 소스와 형식을\n   - 랭체인 프레임워크에서 사용할 수 있도록\n   - 표준화된 문서 형식으로 변환하는 도구임\n\n\n * 80개 이상의 다양한 유형을 지원함\n   - 텍스트, PDF, 웹페이지, CSV 등 다양한 형식의 데이터를 처리할 수 있음\n\n\n * 모든 로더는 데이터를 content와 metadata를 포함하는\n   - 표준 문서 객체로 변환하여 랭체인의 다른 컴포넌트와 쉽게 통합할 수 있음\n\n\n<문서 로더의 클래스(class) 종류>\n * WebBaseLoader\n   - 웹 페이지(HTML 데이터)의 내용을 로드하는 기능을 제공함\n * PyPDFLoader\n   - PDF 파일을 로드하고 필요에 따라 페이지 단위로 분할하는 기능을 제공함\n * CSVLoader\n   - CSV 파일을 로드하는 기능을 제공함\n * DirectoryLoader\n   - 지정된 디렉토리 내의 모든 파일을 로드하는 기능을 제공함\n * UnstructuredLoader\n   - 다양한 형식의 파일을 로드하는 기능을 제공함\n\n\n<문서 로더 라이브러리 설치>

### 웹 페이지 읽어들이기(WebBaseLoader)

#### 라이브러리 정의

In [2]:
"""
* 웹페이지 로더 라이브러리를 설치후
 - 단독으로, from langchain_community.document_loaders import WebBaseLoader 불러들일 경우
 - 아래 <경고> 메시지가 나타남
 
<경고 내용>
 - USER_AGENT environment variable not set, consider setting it to identify your requests.
 - USER_AGENT 환경 변수가 설정되지 않았습니다. 요청을 식별하기 위해 설정하는 것이 좋습니다.


<경고가 발생한 이유>
 - 웹 페이지를 로드하기 위해서는 현재 프로그램 영역에서
   외부 웹페이지(웹서버) 영역으로 요청(URL)을 보내 후
   응답(페이지 정보)을 받아오게 됨
 - 외부로 요청을 보내는 일련의 과정을 OS에 식별자로 정의해야
   요청에 대한 추적이 용이하다는 의견임


<해결 방법>
 - 경고 메시지의 "USER_AGENT"라는 이름으로 환경변수에 설정하라는 의미임
 - 즉, 사용자 에이전트 설정을 하라는 의미임  
"""


'\n* 웹페이지 로더 라이브러리를 설치후\n - 단독으로, from langchain_community.document_loaders import WebBaseLoader 불러들일 경우\n - 아래 <경고> 메시지가 나타남\n \n<경고 내용>\n - USER_AGENT environment variable not set, consider setting it to identify your requests.\n - USER_AGENT 환경 변수가 설정되지 않았습니다. 요청을 식별하기 위해 설정하는 것이 좋습니다.\n\n\n<경고가 발생한 이유>\n - 웹 페이지를 로드하기 위해서는 현재 프로그램 영역에서\n   외부 웹페이지(웹서버) 영역으로 요청(URL)을 보내 후\n   응답(페이지 정보)을 받아오게 됨\n - 외부로 요청을 보내는 일련의 과정을 OS에 식별자로 정의해야\n   요청에 대한 추적이 용이하다는 의견임\n\n\n<해결 방법>\n - 경고 메시지의 "USER_AGENT"라는 이름으로 환경변수에 설정하라는 의미임\n - 즉, 사용자 에이전트 설정을 하라는 의미임  \n'

In [3]:
### 사용 운영체제 라이브러리
import os

### 웹페이지 정보수집을 위한 에이전트 설정
# - 웹페이지 로더 라이브러리를 사용하기 위해서는 사전에 에이전트를 설정할 필요 있음
os.environ["USER_AGENT"] = "MyApp/1.0 (Custom LangChain Application)"

### 웹페이지 로더 라이브러리
from langchain_community.document_loaders import WebBaseLoader

/tmp/ipykernel_3117/572115688.py:9: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import WebBaseLoader


### 단일 URL 요청 처리하기

In [4]:
### 요청 URL 변수로 정의
url = "https://docs.smith.langchain.com/"

### 웹페이지 로더 클래스 생성하기 : 요청 url을 넣어 주셔야 합니다.
loader_single_page = WebBaseLoader(url)
loader_single_page

### 단일 웹페이지 URL 정보에 대한 페이지 정보 로드(읽어들이기)
# - 반환 타입은 리스트(list) 타입
single_doc = loader_single_page.load()
single_doc

[Document(metadata={'source': 'https://docs.smith.langchain.com/', 'title': 'LangSmith Observability - Docs by LangChain', 'description': 'Instrument your LLM application, investigate traces, and monitor performance in production with LangSmith.', 'language': 'en'}, page_content='\n\n\n\n\nLangSmith Observability - Docs by LangChain\n\n\n\n\n\n\n\n\n\nDocumentation IndexFetch the complete documentation index at: /llms.txtUse this file to discover all available pages before exploring further.Skip to main contentInterrupt is coming to San Francisco March 24–25 | Get early bird pricing nowDocs by LangChain home pageMonitorSearch...⌘KAsk AIGitHubTry LangSmithTry LangSmithSearch...NavigationLangSmith ObservabilityOverviewTraceDebugObserveReferenceLangSmith ObservabilityLangSmith Observability provides full visibility into your LLM application: from individual traces to production-wide performance metrics. Traces are the record of what your agents did in production. Use them to debug failure

In [5]:
### 읽어들인 웹페이지 정보는 리스트 타입으로 되어있음
# - 따라서, 리스트 안으로 들어가서 데이터 추출하기
single_doc[0]

### 읽어들인 정보에는 metadata와 page_content 속성 2개가 있음
# - metadata : 웹페이지 기본 정보가 들어있음(딕셔너리 타입으로 정보가 있음)
# - page_content : 웹페이지의 본문 내용이 들어있음

Document(metadata={'source': 'https://docs.smith.langchain.com/', 'title': 'LangSmith Observability - Docs by LangChain', 'description': 'Instrument your LLM application, investigate traces, and monitor performance in production with LangSmith.', 'language': 'en'}, page_content='\n\n\n\n\nLangSmith Observability - Docs by LangChain\n\n\n\n\n\n\n\n\n\nDocumentation IndexFetch the complete documentation index at: /llms.txtUse this file to discover all available pages before exploring further.Skip to main contentInterrupt is coming to San Francisco March 24–25 | Get early bird pricing nowDocs by LangChain home pageMonitorSearch...⌘KAsk AIGitHubTry LangSmithTry LangSmithSearch...NavigationLangSmith ObservabilityOverviewTraceDebugObserveReferenceLangSmith ObservabilityLangSmith Observability provides full visibility into your LLM application: from individual traces to production-wide performance metrics. Traces are the record of what your agents did in production. Use them to debug failures

In [6]:
### metadata(웹페이지 기본 정보) 확인하기
meta_dict = single_doc[0].metadata

### metadata 내에 있는 모든 정보를 출력(print) 하기
for k, v in meta_dict.items():
    print(f" [{k}] : {v}")


# 딕셔너리에서 키값만 출력하기
print(f"딕셔너리의 key값만 추출하기 : {meta_dict.keys()}")
# 딕셔너리에서 밸류값만 출력하기
print(f"딕셔너리의 value값만 추출하기 : {meta_dict.values()}")
# 딕셔너리에서 키값 밸류값 모두 출력하기
print(f"딕셔너리의 키값 밸류값 추출하기 : {meta_dict.items()}")

 [source] : https://docs.smith.langchain.com/
 [title] : LangSmith Observability - Docs by LangChain
 [description] : Instrument your LLM application, investigate traces, and monitor performance in production with LangSmith.
 [language] : en
딕셔너리의 key값만 추출하기 : dict_keys(['source', 'title', 'description', 'language'])
딕셔너리의 value값만 추출하기 : dict_values(['https://docs.smith.langchain.com/', 'LangSmith Observability - Docs by LangChain', 'Instrument your LLM application, investigate traces, and monitor performance in production with LangSmith.', 'en'])
딕셔너리의 키값 밸류값 추출하기 : dict_items([('source', 'https://docs.smith.langchain.com/'), ('title', 'LangSmith Observability - Docs by LangChain'), ('description', 'Instrument your LLM application, investigate traces, and monitor performance in production with LangSmith.'), ('language', 'en')])


In [7]:
### 수집한 페이지 본문 내용 확인하기
# - 본문 내용은 page_content 속성에 문자열로 저장되어 있음
# - page_content 속성(변수)에 저장된 값 출력해주세요...
single_doc

[Document(metadata={'source': 'https://docs.smith.langchain.com/', 'title': 'LangSmith Observability - Docs by LangChain', 'description': 'Instrument your LLM application, investigate traces, and monitor performance in production with LangSmith.', 'language': 'en'}, page_content='\n\n\n\n\nLangSmith Observability - Docs by LangChain\n\n\n\n\n\n\n\n\n\nDocumentation IndexFetch the complete documentation index at: /llms.txtUse this file to discover all available pages before exploring further.Skip to main contentInterrupt is coming to San Francisco March 24–25 | Get early bird pricing nowDocs by LangChain home pageMonitorSearch...⌘KAsk AIGitHubTry LangSmithTry LangSmithSearch...NavigationLangSmith ObservabilityOverviewTraceDebugObserveReferenceLangSmith ObservabilityLangSmith Observability provides full visibility into your LLM application: from individual traces to production-wide performance metrics. Traces are the record of what your agents did in production. Use them to debug failure

In [8]:
text = single_doc[0].page_content
print(text)






LangSmith Observability - Docs by LangChain









Documentation IndexFetch the complete documentation index at: /llms.txtUse this file to discover all available pages before exploring further.Skip to main contentInterrupt is coming to San Francisco March 24–25 | Get early bird pricing nowDocs by LangChain home pageMonitorSearch...⌘KAsk AIGitHubTry LangSmithTry LangSmithSearch...NavigationLangSmith ObservabilityOverviewTraceDebugObserveReferenceLangSmith ObservabilityLangSmith Observability provides full visibility into your LLM application: from individual traces to production-wide performance metrics. Traces are the record of what your agents did in production. Use them to debug failures, monitor quality, and build the datasets you evaluate against.LangSmith works with many frameworks and providers. Browse available integrations to connect your stack including OpenAI, Anthropic, CrewAI, Vercel AI SDK, Pydantic AI, and more.Get startedCreate an accountSign up at smith.langchain

### 새로운 URL 웹페이지 데이터 수집하기

In [9]:
### 새로운 URL : https://www.ibm.com/kr-ko/think/topics/machine-learning
# - 웹페이지 본문 내용 추출하여 "text 변수에 저장"하여 출력해 주세요..

In [10]:
### 임의 URL을 전달 받아서, 해당 URL 웹페이지의 본문 내용을 추출하여 반환하는 
# - 함수 정의하기 : 함수 이름은 getPageContent
# - 함수로부터 반환받은 본문 내용 잘 출력되는지 확인까지 해주세요..
def getPageContent(get_url):
    url = get_url

    loader_single_page = WebBaseLoader(url)

    single_doc = loader_single_page.load()

    text = single_doc[0].page_content

    return text

In [11]:
gPC = getPageContent("https://www.ibm.com/kr-ko/think/topics/machine-learning")

print(gPC)
















머신 러닝이란 무엇인가요? | IBM 









































































































AWS re:Invent에서 IBM과 함께하세요
IBM이 AWS에서 엔터프라이즈급 제어 기능을 유지하면서 AI의 속도를 구현하는 방법을 알아보세요

































                    


  
  
    
    머신 러닝이란 무엇인가요?

  





    


                



                    게시일 2025년 08월 18일
                

                    수정일 2026년 10월 2일
                

















                By 
                Dave Bergmann










        머신 러닝이란 무엇인가요?
    


머신 러닝은 학습 데이터의 패턴을 "학습"한 후 새로운 데이터에 대해 정확한 추론을 할 수 있는 알고리즘에 초점을 맞춘 인공 지능(AI)의 하위 집합입니다. 이 패턴 인식 기능을 통해 머신 러닝 모델은 명시적이고 하드 코딩된 지침 없이 결정이나 예측을 내릴 수 있습니다.


머신 러닝은 예측 모델부터 자율 주행 차량, 대규모 언어 모델(LLM) 및 기타 생성형 AI 도구에 이르기까지 대부분의 최신 AI 시스템의 중추를 제공하는 AI 분야를 지배하게 되었습니다.
머신 러닝(ML)의 핵심적인 전제는 모델 학습이라는 프로세스를 통해 모델이 사용될 실제 문제와 충분히 유사한 작업 데이터 세트에서 모델의 성능을 최적화하면 모델이 궁극적인 사용 사례에서 보게 되는 새 데이터를 정확하게 예측할 수 있다는 것입니다.
학습 자체는 단순히 목적을 위한 수단일 뿐이며, 학습 데이터의 성능을 

In [12]:
print(text)






LangSmith Observability - Docs by LangChain









Documentation IndexFetch the complete documentation index at: /llms.txtUse this file to discover all available pages before exploring further.Skip to main contentInterrupt is coming to San Francisco March 24–25 | Get early bird pricing nowDocs by LangChain home pageMonitorSearch...⌘KAsk AIGitHubTry LangSmithTry LangSmithSearch...NavigationLangSmith ObservabilityOverviewTraceDebugObserveReferenceLangSmith ObservabilityLangSmith Observability provides full visibility into your LLM application: from individual traces to production-wide performance metrics. Traces are the record of what your agents did in production. Use them to debug failures, monitor quality, and build the datasets you evaluate against.LangSmith works with many frameworks and providers. Browse available integrations to connect your stack including OpenAI, Anthropic, CrewAI, Vercel AI SDK, Pydantic AI, and more.Get startedCreate an accountSign up at smith.langchain

In [13]:
url = "https://www.ibm.com/kr-ko/think/topics/machine-learning"
url

load_page = WebBaseLoader(url)
load_page

load_doc = load_page.load()
load_doc

text = load_doc[0].page_content
# print(text)
text

'\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n머신 러닝이란 무엇인가요? | IBM \n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\nAWS re:Invent에서 IBM과 함께하세요\nIBM이 AWS에서 엔터프라이즈급 제어 기능을 유지하면서 AI의 속도를 구현하는 방법을 알아보세요\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n                    \n\n\n  \n  \n    \n    머신 러닝이란 무엇인가요?\n\n  \n\n\n\n\n\n    \n\n\n                \n\n\n\n                    게시일 2025년 08월 18일\n                \n\n                    수정일 2026년 10월 2일\n                \n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n                By \n                Dave Bergmann\n\n\n\n\n\n\n\n\n\n\n        머신 러닝이란 무엇인가요?\n    \n\n\n머신 러닝은 학습 데이터의 패턴을 "학습"한 후 새로운 데이터에 대해 정확한 추론을 할 수 있는 알고리즘에 초점을 맞춘 인공 지능(AI)의 하위 집합입니다. 이 패턴 인식 기능을 통해 머신 러닝 모델은 명시적이고 하드 코딩된 지침 없이 결정이나 예측을 내릴 수 있습니다.\n\n\n머신 러닝은 예측 모델부터 자율 주행\xa0차량, 대규모 언어 모델(LLM) 및 기타\xa0생성형 AI 도구에 이르

In [14]:
text

'\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n머신 러닝이란 무엇인가요? | IBM \n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\nAWS re:Invent에서 IBM과 함께하세요\nIBM이 AWS에서 엔터프라이즈급 제어 기능을 유지하면서 AI의 속도를 구현하는 방법을 알아보세요\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n                    \n\n\n  \n  \n    \n    머신 러닝이란 무엇인가요?\n\n  \n\n\n\n\n\n    \n\n\n                \n\n\n\n                    게시일 2025년 08월 18일\n                \n\n                    수정일 2026년 10월 2일\n                \n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n                By \n                Dave Bergmann\n\n\n\n\n\n\n\n\n\n\n        머신 러닝이란 무엇인가요?\n    \n\n\n머신 러닝은 학습 데이터의 패턴을 "학습"한 후 새로운 데이터에 대해 정확한 추론을 할 수 있는 알고리즘에 초점을 맞춘 인공 지능(AI)의 하위 집합입니다. 이 패턴 인식 기능을 통해 머신 러닝 모델은 명시적이고 하드 코딩된 지침 없이 결정이나 예측을 내릴 수 있습니다.\n\n\n머신 러닝은 예측 모델부터 자율 주행\xa0차량, 대규모 언어 모델(LLM) 및 기타\xa0생성형 AI 도구에 이르

In [15]:
text.count("머신")

54

### 수집 데이터 전처리(정규표현식-Regular Expression )

In [16]:
### text 데이터에서 \n, 문장의 양끝 공백, 탭 등 제거해 주세요...
text = text.strip()
# text = text.replace("\t", "")
# while True:
#     text = text.replace("\n", "")
    

print(text)

머신 러닝이란 무엇인가요? | IBM 









































































































AWS re:Invent에서 IBM과 함께하세요
IBM이 AWS에서 엔터프라이즈급 제어 기능을 유지하면서 AI의 속도를 구현하는 방법을 알아보세요

































                    


  
  
    
    머신 러닝이란 무엇인가요?

  





    


                



                    게시일 2025년 08월 18일
                

                    수정일 2026년 10월 2일
                

















                By 
                Dave Bergmann










        머신 러닝이란 무엇인가요?
    


머신 러닝은 학습 데이터의 패턴을 "학습"한 후 새로운 데이터에 대해 정확한 추론을 할 수 있는 알고리즘에 초점을 맞춘 인공 지능(AI)의 하위 집합입니다. 이 패턴 인식 기능을 통해 머신 러닝 모델은 명시적이고 하드 코딩된 지침 없이 결정이나 예측을 내릴 수 있습니다.


머신 러닝은 예측 모델부터 자율 주행 차량, 대규모 언어 모델(LLM) 및 기타 생성형 AI 도구에 이르기까지 대부분의 최신 AI 시스템의 중추를 제공하는 AI 분야를 지배하게 되었습니다.
머신 러닝(ML)의 핵심적인 전제는 모델 학습이라는 프로세스를 통해 모델이 사용될 실제 문제와 충분히 유사한 작업 데이터 세트에서 모델의 성능을 최적화하면 모델이 궁극적인 사용 사례에서 보게 되는 새 데이터를 정확하게 예측할 수 있다는 것입니다.
학습 자체는 단순히 목적을 위한 수단일 뿐이며, 학습 데이터의 성능을 실제 시나리오에서 유용한 결

In [17]:
### 정규표현식(re) 라이브러리
# - 문자열 내에서 특정 패턴을 이용하여 수정/제거하는 표현식(특정 패턴)을 의미함
import re

### 주로 사용되는 함수(문자열 함수)
# - re.sub(패턴, 바꿀문자열, 원본): 원본 문자열에서 패턴에 해당하는 부분을 지정한 문자로 일괄 치환
# - strip() : 문자열 맨 앞과 맨 뒤에 남아있는 의미 없는 공백을 제거
# - split(기호) : 문자열 내에서 특정 기호를 이용하여 분리하고자 할 때 사용
#                 (분리된 데이터는 리스트 타입으로 관리됨)

# str = "안녕, 너는, 누구니, ??"
# str_split = str.split(", ") # `, `를 기준으로 문자열로 분리 -> 리스트
# str_split

### 1. 탭(\t)과 연속된 띄어쓰기(" ")를 하나의 공백(" ")으로 치환
# - 탭과 연속된(+ 의미) 띄어쓰기 패턴 : [ \t]+
# - re.sub(패턴, 바꿀문자열, 원본)
text_t = re.sub(r"[ \t]+", " ", text)


### 2. 2개 이상의 줄바꿈(\n\n\n\n\n\n\n\n\n)을 => 2개의 줄바꿈(\n\n)으로 변환
# - "\n{2,}"
text_t = re.sub(r"\n{2,}", "\n\n",text_t)
# print(text_t)

### 3. 줄바꿈(\n) 기호를 이용하여 분리하고, 양쪽 문장의 끝 공백을 제거하기
# - 줄바꿈(\n) 기호를 이용하여 분리 : "원본 텍스트"split("\n") -> 반환 타입 `리스트`
# - 양쪽 문장의 끝 공백을 제거 : "각각 분리된 문장".strip()
# - 리스트 타입으로 정의 : 한줄 코딩(컴프리헨션 코딩)

text_list = [txt.strip() for txt in text_t.split("\n")]

### 4. 분리된 리스트 내의 각 인덱스에 있는 문장들을 하나의 문장으로 모두 합치기
# - 각 인덱스마다 \n을 넣어서 합치기
# - "합치기 조건 기호".join(원본) : 리스트 내의 각 인덱스의 값을 하나로 합칠 때 사용하는 함수
text_all = "\n".join(text_list)

### 5. 3개 이상의 줄바꿈(\n)을 -> 두 개의 줄바꿈(\n\n)으로 치환(변환)하고 -> 양끝 공백 제거
cleaned_text = re.sub(r"\n{3,}","\n\n",text_all).strip()

print(cleaned_text)
cleaned_text



머신 러닝이란 무엇인가요? | IBM

AWS re:Invent에서 IBM과 함께하세요
IBM이 AWS에서 엔터프라이즈급 제어 기능을 유지하면서 AI의 속도를 구현하는 방법을 알아보세요

머신 러닝이란 무엇인가요?

게시일 2025년 08월 18일

수정일 2026년 10월 2일

By
Dave Bergmann

머신 러닝이란 무엇인가요?

머신 러닝은 학습 데이터의 패턴을 "학습"한 후 새로운 데이터에 대해 정확한 추론을 할 수 있는 알고리즘에 초점을 맞춘 인공 지능(AI)의 하위 집합입니다. 이 패턴 인식 기능을 통해 머신 러닝 모델은 명시적이고 하드 코딩된 지침 없이 결정이나 예측을 내릴 수 있습니다.

머신 러닝은 예측 모델부터 자율 주행 차량, 대규모 언어 모델(LLM) 및 기타 생성형 AI 도구에 이르기까지 대부분의 최신 AI 시스템의 중추를 제공하는 AI 분야를 지배하게 되었습니다.
머신 러닝(ML)의 핵심적인 전제는 모델 학습이라는 프로세스를 통해 모델이 사용될 실제 문제와 충분히 유사한 작업 데이터 세트에서 모델의 성능을 최적화하면 모델이 궁극적인 사용 사례에서 보게 되는 새 데이터를 정확하게 예측할 수 있다는 것입니다.
학습 자체는 단순히 목적을 위한 수단일 뿐이며, 학습 데이터의 성능을 실제 시나리오에서 유용한 결과로 변환하는 일반화는 머신 러닝의 기본 목표입니다. 본질적으로 학습된 모델은 학습 데이터에서 학습한 패턴을 적용하여 실제 작업에 대한 올바른 아웃풋을 추론하는 것입니다. 따라서 AI 모델의 배포를 AI 추론이라고 합니다.
대규모 또는 '심층' 인공 신경망에 의해 구동되는 머신 러닝의 하위 집합인딥 러닝은 지난 수십 년 동안 AI가 활용되는 거의 모든 영역에서 최첨단 AI 모델 아키텍처로 등장했습니다. 기존 머신 러닝의 명시적으로 정의된 알고리즘과 달리, 딥 러닝은 매우 복잡한 데이터의 복잡한 뉘앙스를 학습할 수 있는 최상의 능력을 제공하는 분산된 수학 연산 "네트워크"에 의존합니다. 딥 러닝에는 매우 많은 양의 데이터와 계산 리소

'머신 러닝이란 무엇인가요? | IBM\n\nAWS re:Invent에서 IBM과 함께하세요\nIBM이 AWS에서 엔터프라이즈급 제어 기능을 유지하면서 AI의 속도를 구현하는 방법을 알아보세요\n\n머신 러닝이란 무엇인가요?\n\n게시일 2025년 08월 18일\n\n수정일 2026년 10월 2일\n\nBy\nDave Bergmann\n\n머신 러닝이란 무엇인가요?\n\n머신 러닝은 학습 데이터의 패턴을 "학습"한 후 새로운 데이터에 대해 정확한 추론을 할 수 있는 알고리즘에 초점을 맞춘 인공 지능(AI)의 하위 집합입니다. 이 패턴 인식 기능을 통해 머신 러닝 모델은 명시적이고 하드 코딩된 지침 없이 결정이나 예측을 내릴 수 있습니다.\n\n머신 러닝은 예측 모델부터 자율 주행\xa0차량, 대규모 언어 모델(LLM) 및 기타\xa0생성형 AI 도구에 이르기까지 대부분의 최신 AI 시스템의 중추를 제공하는 AI 분야를 지배하게 되었습니다.\n머신 러닝(ML)의 핵심적인 전제는 모델 학습이라는 프로세스를 통해 모델이 사용될 실제 문제와 충분히 유사한 작업 데이터 세트에서 모델의 성능을 최적화하면 모델이 궁극적인 사용 사례에서 보게 되는 새 데이터를 정확하게 예측할 수 있다는 것입니다.\n학습 자체는 단순히 목적을 위한 수단일 뿐이며, 학습 데이터의 성능을 실제 시나리오에서 유용한 결과로 변환하는 일반화는 머신 러닝의 기본 목표입니다.\xa0본질적으로 학습된 모델은 학습 데이터에서 학습한 패턴을 적용하여 실제 작업에 대한 올바른 아웃풋을 추론하는 것입니다. 따라서 AI 모델의 배포를\xa0AI 추론이라고 합니다.\n대규모 또는 \'심층\' 인공 신경망에 의해 구동되는 머신 러닝의 하위 집합인딥 러닝은 지난 수십 년 동안 AI가 활용되는 거의 모든 영역에서 최첨단 AI 모델 아키텍처로 등장했습니다. 기존 머신 러닝의 명시적으로 정의된 알고리즘과 달리, 딥 러닝은 매우 복잡한 데이터의 복잡한 뉘앙스를 학습할 수 있는 최상의 능력을 제공하는 분산된 수학 연산 "네트워크

In [18]:
### 웹 수집 데이터 전처리 함수
def getRegular_Expression(p_text): 
    ### 1. 탭(\t)과 연속된 띄어쓰기(" ")를 하나의 공백(" ")으로 치환
    # - 탭과 연속된(+ 의미) 띄어쓰기 패턴 : [ \t]+
    # - re.sub(패턴, 바꿀문자열, 원본)
    text_t = re.sub(r"[ \t]+", " ", p_text)


    ### 2. 2개 이상의 줄바꿈(\n\n\n\n\n\n\n\n\n)을 => 2개의 줄바꿈(\n\n)으로 변환
    # - "\n{2,}"
    text_t = re.sub(r"\n{2,}", "\n\n",text_t)
    # print(text_t)

    ### 3. 줄바꿈(\n) 기호를 이용하여 분리하고, 양쪽 문장의 끝 공백을 제거하기
    # - 줄바꿈(\n) 기호를 이용하여 분리 : "원본 텍스트"split("\n") -> 반환 타입 `리스트``
    # - 양쪽 문장의 끝 공백을 제거 : "각각 분리된 문장".strip()
    # - 리스트 타입으로 정의 : 한줄 코딩(컴프리헨션 코딩)

    text_list = [txt.strip() for txt in text_t.split("\n")]

    ### 4. 분리된 리스트 내의 각 인덱스에 있는 문장들을 하나의 문장으로 모두 합치기
    # - 각 인덱스마다 \n을 넣어서 합치기
    # - "합치기 조건 기호".join(원본) : 리스트 내의 각 인덱스의 값을 하나로 합칠 때 사용하는 함수
    text_all = "\n".join(text_list)

    ### 5. 3개 이상의 줄바꿈(\n)을 -> 두 개의 줄바꿈(\n\n)으로 치환(변환)하고 -> 양끝 공백 제거
    cleaned_text = re.sub(r"\n{3,}","\n\n",text_all).strip()

    ### 6. 처리결과 반환하기
    return cleaned_text


In [19]:
"""
 * URL "https://www.ibm.com/kr-ko/think/topics/machine-learning"의 수집한 웹페이지 정보에서
  - 페이지 제목(title), 요약 내용(description), 본문 내용(전처리 완료된 내용)을
    출력(print)해 주세요..
  - 웹페이지 읽어들이기부터 진행(처음부터)
"""
### url 주소
url = "https://www.ibm.com/kr-ko/think/topics/machine-learning"

### 페이지 불러들이기
loader_single_page = WebBaseLoader(url)
loader_single_page

### 페이지 내 텍스트 불러들이기
load_doc = loader_single_page.load()

meta_dict = load_doc[0].metadata
page_content = load_doc[0].page_content

### 본문 내용 전처리
# - 연속된 탭과 공백을 공백으로 치환
text = re.sub("[ \t]+" , " ",page_content)
text

# - 2개 이상의 \n을 2개 \n으로 치환하기
text_n = re.sub("\n{2,}", "\n\n", text)
text_n

# - \n을 기준으로 분리하여 양쪽 끝 공백 제거 후 리스트에 담기
text_list = [txt.strip() for txt in text_n.split("\n")]
text_list

# - 각 인덱스마다 \n을 넣어 하나의 문자열로 다시 합치기
text_all = "\n".join(text_list)
text_all

# - 양쪽 공백 제거 및 2개 이상의 \n을 2개의 \n으로 바꾸기
text_final = re.sub("\n{2,}", "\n\n", text_all).strip()
text_final

print(f"[제목] : {meta_dict['title']}")
print(f"[요약 내용] : {meta_dict['description']}")
print(f"[본문 내용] : {text_final}")

[제목] : 머신 러닝이란 무엇인가요? | IBM 
[요약 내용] : 머신 러닝은 새로운 데이터에 대한 정확한 추론을 하기 위해 학습 데이터의 패턴을 분석하고 "학습"하는 알고리즘에 초점을 맞춘 AI의 하위 집합입니다. 
[본문 내용] : 머신 러닝이란 무엇인가요? | IBM

AWS re:Invent에서 IBM과 함께하세요
IBM이 AWS에서 엔터프라이즈급 제어 기능을 유지하면서 AI의 속도를 구현하는 방법을 알아보세요

머신 러닝이란 무엇인가요?

게시일 2025년 08월 18일

수정일 2026년 10월 2일

By
Dave Bergmann

머신 러닝이란 무엇인가요?

머신 러닝은 학습 데이터의 패턴을 "학습"한 후 새로운 데이터에 대해 정확한 추론을 할 수 있는 알고리즘에 초점을 맞춘 인공 지능(AI)의 하위 집합입니다. 이 패턴 인식 기능을 통해 머신 러닝 모델은 명시적이고 하드 코딩된 지침 없이 결정이나 예측을 내릴 수 있습니다.

머신 러닝은 예측 모델부터 자율 주행 차량, 대규모 언어 모델(LLM) 및 기타 생성형 AI 도구에 이르기까지 대부분의 최신 AI 시스템의 중추를 제공하는 AI 분야를 지배하게 되었습니다.
머신 러닝(ML)의 핵심적인 전제는 모델 학습이라는 프로세스를 통해 모델이 사용될 실제 문제와 충분히 유사한 작업 데이터 세트에서 모델의 성능을 최적화하면 모델이 궁극적인 사용 사례에서 보게 되는 새 데이터를 정확하게 예측할 수 있다는 것입니다.
학습 자체는 단순히 목적을 위한 수단일 뿐이며, 학습 데이터의 성능을 실제 시나리오에서 유용한 결과로 변환하는 일반화는 머신 러닝의 기본 목표입니다. 본질적으로 학습된 모델은 학습 데이터에서 학습한 패턴을 적용하여 실제 작업에 대한 올바른 아웃풋을 추론하는 것입니다. 따라서 AI 모델의 배포를 AI 추론이라고 합니다.
대규모 또는 '심층' 인공 신경망에 의해 구동되는 머신 러닝의 하위 집합인딥 러닝은 지난 수십 년 동안 AI가 활용되는 거의 모든 영역에서 최첨단 AI 모델 아키텍처로 등장했습

### 다중 URL을 이용하여 웹페이지 정보 수집하기

In [20]:
### 다중 URL을 사용하기 위해서는 URL을 리스트 타입으로 정의
# - 여러개 값을 저장하는 타입 : 리스트, 딕셔너리, 튜플
urls = ["https://docs.smith.langchain.com/",
        "https://langchain-ai.github.io/langgraph"]

### 1. 웹페이지로드 클래스 생성하기
loader_multiple_page = WebBaseLoader(urls)
loader_multiple_page

### 2. 다중 URL을 이용하여 각각의 웹페이지 정보 로드하기(읽어 들이기)
multiple_doc = loader_multiple_page.load()
multiple_doc

[Document(metadata={'source': 'https://docs.smith.langchain.com/', 'title': 'LangSmith Observability - Docs by LangChain', 'description': 'Instrument your LLM application, investigate traces, and monitor performance in production with LangSmith.', 'language': 'en'}, page_content='\n\n\n\n\nLangSmith Observability - Docs by LangChain\n\n\n\n\n\n\n\n\n\nDocumentation IndexFetch the complete documentation index at: /llms.txtUse this file to discover all available pages before exploring further.Skip to main contentInterrupt is coming to San Francisco March 24–25 | Get early bird pricing nowDocs by LangChain home pageMonitorSearch...⌘KAsk AIGitHubTry LangSmithTry LangSmithSearch...NavigationLangSmith ObservabilityOverviewTraceDebugObserveReferenceLangSmith ObservabilityLangSmith Observability provides full visibility into your LLM application: from individual traces to production-wide performance metrics. Traces are the record of what your agents did in production. Use them to debug failure

In [21]:

multiple_doc[1].metadata["title"]

'Redirecting to LangGraph Documentation'

In [22]:
### 딕셔너리 데이터 추출 방법(2가지)
dits = {"name" : "홍길동", "age" : 44}

# - 첫번째 방법 : 기존과 동일
dits["name"]

# - 두번째 방법 : get(key, "key가 존재하지 않을 때 사용할 값 정의") 함수 사용
dits.get("name2", "key가 존재하지 않음")

'key가 존재하지 않음'

In [23]:
### *각각의 웹페이지 정보에 대해서
# - 페이지 제목, 요약내용, 본문 내용을
#   출력해 주세요..


### for문 사용 방법1 : 리스트의 인덱스 번호를 이용하여 접근하는 방법
for index in range(0, len(multiple_doc), 1):
    ### 제목
    # title = multiple_doc[index].metadata["title"]
    title = multiple_doc[index].metadata.get("title", "title이 존재하지 않습니다.")

    description = multiple_doc[index].metadata.get("description", "요약 내용 없음")

    ### 본문 내용
    content = multiple_doc[index].page_content

    print(f"[제목] : {title} [요약 내용] : {description}\n[본문 내용] : {content}\n")



### for문 사용 방법2 : 리스트의 값을 이용하여 접근하는 방법
for doc in multiple_doc:
    ### 제목
    title = doc.metadata["title"]


    ### 요약 내용, get 함수 (key를 주고 value를 주는데, 키가 없을 경우 None 처리를 하여 반환함.)
    # - 딕셔너리.get("value값을 받아올 key값", "key값이 존재하지 않을 때 반환할 값")
    description = doc.metadata.get("description", "요약 내용 없음")

    ### 본문 내용
    content = doc.page_content

    print(f"[제목] : {title} [요약 내용] : {description}\n[본문 내용] : {content}\n")




[제목] : LangSmith Observability - Docs by LangChain [요약 내용] : Instrument your LLM application, investigate traces, and monitor performance in production with LangSmith.
[본문 내용] : 




LangSmith Observability - Docs by LangChain









Documentation IndexFetch the complete documentation index at: /llms.txtUse this file to discover all available pages before exploring further.Skip to main contentInterrupt is coming to San Francisco March 24–25 | Get early bird pricing nowDocs by LangChain home pageMonitorSearch...⌘KAsk AIGitHubTry LangSmithTry LangSmithSearch...NavigationLangSmith ObservabilityOverviewTraceDebugObserveReferenceLangSmith ObservabilityLangSmith Observability provides full visibility into your LLM application: from individual traces to production-wide performance metrics. Traces are the record of what your agents did in production. Use them to debug failures, monitor quality, and build the datasets you evaluate against.LangSmith works with many frameworks and providers. Br

### PDF파일(문서) 로더하기(읽어들이기)-PyPDFLoader 클래스 사용

In [24]:
"""
<PDF 파일 로더(PyPDFLoader)>
 * PDF 문서를 페이지 단위로 로드하는데 자주 사용되는 도구임


 * PDF 문서 내의 구조에 따른 사용 라이브러리
  - PyPDFLoader
    : 간단하고 사용하기 쉬움
    : 문서 내의 구조가 이미지나 표가 포함된 복잡한 구조에서는 정확도가 떨어짐


  - PyMuPDFLoader
    : 빠르고 정확한 텍스트 추출이 가능하며,
    : 이미지, 표, 레이아웃 정보 등을 추출할 수 있음


  - PDFPlumberLoader
    : 표와 같은 구조화된 데이터 추출에 특화되어 있음


<PDF 파일 로드를 위해 라이브러리 추가 설치>
  >uv pip install pypdf pymupdf pdfplumber
"""



'\n<PDF 파일 로더(PyPDFLoader)>\n * PDF 문서를 페이지 단위로 로드하는데 자주 사용되는 도구임\n\n\n * PDF 문서 내의 구조에 따른 사용 라이브러리\n  - PyPDFLoader\n    : 간단하고 사용하기 쉬움\n    : 문서 내의 구조가 이미지나 표가 포함된 복잡한 구조에서는 정확도가 떨어짐\n\n\n  - PyMuPDFLoader\n    : 빠르고 정확한 텍스트 추출이 가능하며,\n    : 이미지, 표, 레이아웃 정보 등을 추출할 수 있음\n\n\n  - PDFPlumberLoader\n    : 표와 같은 구조화된 데이터 추출에 특화되어 있음\n\n\n<PDF 파일 로드를 위해 라이브러리 추가 설치>\n  >uv pip install pypdf pymupdf pdfplumber\n'

### PDF 로더 라이브러리 불러들이기

In [25]:
from langchain_community.document_loaders import PyMuPDFLoader
from langchain_community.document_loaders import PyPDFLoader
from langchain_community.document_loaders import PDFPlumberLoader

In [26]:
"""
- PDF 로더시 다소 시간이 많이 걸립니다.
- 프로그램 실행 시간 확인 방법
- 각 프로그램 가장 상단에 %%time 명령어를 작성해 놓으면,
  -> 실행이 끝난 후 실행 시간을 함께 출력해 줍니다.
  -> 주피터 노트북에서 사용하는 옵션으로, 순수 Python 프로그램에서는 사용 못함
"""


'\n- PDF 로더시 다소 시간이 많이 걸립니다.\n- 프로그램 실행 시간 확인 방법\n- 각 프로그램 가장 상단에 %%time 명령어를 작성해 놓으면,\n  -> 실행이 끝난 후 실행 시간을 함께 출력해 줍니다.\n  -> 주피터 노트북에서 사용하는 옵션으로, 순수 Python 프로그램에서는 사용 못함\n'

### PyPDFLoader 사용하여 pdf 파일 읽어들이기

In [27]:
%%time
### 실행시간 체크 time 정의 (가장 위에 정의)

### 읽어들일 pdf 파일 위치 정의
file_path = "./data/2024_KB_부동산_보고서_최종.pdf"

### PyPDFLoader 클래스 생성
loader = PyPDFLoader(file_path)
loader

CPU times: user 44 μs, sys: 4 μs, total: 48 μs
Wall time: 42.7 μs


In [28]:
%%time

### PDF파일 실제 로드 및 페이지 분할하기
# - 페이지 분할의 의미 : pdf 페이지를 의미함
# - 반환 타입 : 리스트 타입
# - 리스트 내에 데이터 갯수 : pdf 페이지 수만큼 각각 페이지의 데이터로 반환됨
pages = loader.load_and_split()
print(len(pages))
pages

83
CPU times: user 4.66 s, sys: 78.8 ms, total: 4.74 s
Wall time: 4.81 s


[Document(metadata={'producer': 'Microsoft® Word 2016', 'creator': 'Microsoft® Word 2016', 'creationdate': '2024-03-04T15:30:01+09:00', 'title': 'Morning Meeting', 'author': '손은경', 'moddate': '2024-03-04T15:30:01+09:00', 'source': './data/2024_KB_부동산_보고서_최종.pdf', 'total_pages': 84, 'page': 1, 'page_label': '2'}, page_content='1 \n2024 KB 부동산 보고서: 2024년 주택시장 진단과 전망 \n \n \n \nExecutive Summary 1 \n\uf06e 2024년 주택시장 하향 안정 전망, 금리와 공급, 그리고 정책 3대 변수 주목 \n주택경기가 상승과 하락을 반복하면서 주택 경기의 불확실성이 확대되고 있으나. 완만한 하향 조정 \n가능성이 큰 상황이다. 매수 수요 위축으로 주택 매매 거래량이 급감하면서 향후 주택 경기에 대한 부정적 \n시각이 팽배하다. 무엇보다 여전히 높은 금리가 부담으로 작용하고 있다. 주택 경기 불황기에 고금리 \n부담은 주택 수요를 크게 위축시킬 수밖에 없기 때문이다. 다만 주택시장의 주요 변수들의 상황에 따라 \n소폭 반등 혹은 하락폭 확대 등의 방향이 정해질 것으로 보인다. \n2024년 주택시장의 주요 변수는 공급과 금리다. 급격하게 위축된 주택 공급이 단기간에 증가하기는 \n쉽지 않으나 정부의 공급 시그널이 지속된다면 일정 부분 해소가 될 가능성이 있다. 무엇보다 금리가 주요 \n변수가 될 것이다. 기준금리 인하 시기와 인하 폭에 따라 주택 수요는 영향을 받을 수밖에 없기 때문이다. \n한편, 수요 위축으로 거래가 급감한 상황에서 실수요자 금융 지원, 관련 규제 완화 등 수요 회복을 위한 \n정부 정책도 중요한 변수가 될 전망이다. \n \n\uf06e 7대 이슈

In [29]:
%%time

### 읽어들인 페이지의 10번째 인덱스부터 5개만 본문 내용 출력해 주세요...
for index in range (10, 15, 1):
    content = pages[index].page_content
    print(f"[본문내용]: {content}")
    print("="*100)



[본문내용]: 5 
2024 KB 부동산 보고서: 2024년 주택시장 진단과 전망 
 
표Ⅰ-1. 공급 확대 정책 주요 내용 
날짜 주요 내용 
2023년 9월 
공공 부문 공급물량 확대: 3기 신도시를 포함한 공급물량 확대 및 조기 공급 방안 마련 
사업 여건 개선: 전매 제한 완화 및 규제 정상화, 조기 인허가 인센티브 및 절차 개선, 공사비 증액 
기준 마련 및 인력 확충, 분양 사업의 임대 사업 전환 촉진 
원활한 자금 지원: PF대출 보증 확대, 중도금 대출 지원 
비아파트 자금 조달 지원 및 규제 개선 
도심 공급 기반 확충: 정비사업 절차 및 소규모 사업 사업성 개선 
2024년 1월 
도심 공급 확대: 재건축·재개발 패스트트랙 도입 및 재건축 부담금 합리화, 1기 신도시 재정비사업의 
신속하고 내실 있는 계획 수립, 소규모 정비·도심 복합 사업 속도 개선 및 자금 지원 강화 
다양한 유형의 주택 공급 확대: 도시·건축 규제 완화 및 세제·금융 지원, 등록 임대 사업 여건 개선 
및 기업형 장기임대 활성화, 신축 매입 약정 확대, 전세 사기 예방 및 피해 지원 
신도시 등 공공주택 공급: 2024년 공공주택 14만 호 이상 공급, 신도시 조성 속도 제고 
건설 경기 활력 회복: PF대출 지원 및 유동성 지원, 공공 임대 참여 지분 조기 매각, 민간 사업장 
공공 인수, 재정 조기 집행 및 민자 사업 확대 
 
자료: 국토교통부 보도자료 요약 
 
■ 2024년 주택시장 3대 변수는 공급과 금리, 그리고 정책 
주택시장이 상승과 하락을 반복하면서 주택 경기 불확실성이 확대되고 있다. 특히 매수 수요 위축으
로 주택 매매 거래량이 급감하면서 향후 주택 경기에 대한 부정적 시각이 팽배하다. 무엇보다 여전히 
높은 금리가 부담으로 작용하고 있다. 주택 경기 불황기에 고금리로 인한 대출 부담은 주택 수요를 크
게 위축시킬 수밖에 없기 때문이다. 
 
그림Ⅰ-11. 매수우위지수 추이  표Ⅰ-2. 주요 금융기관 주택담보대출 금리(2024년 2월) 
 
 
상품

In [30]:
%%time

for doc in pages[10:15]:
    content = doc.page_content
    print(f"[본문내용]: {content}")
    print("="*100)

[본문내용]: 5 
2024 KB 부동산 보고서: 2024년 주택시장 진단과 전망 
 
표Ⅰ-1. 공급 확대 정책 주요 내용 
날짜 주요 내용 
2023년 9월 
공공 부문 공급물량 확대: 3기 신도시를 포함한 공급물량 확대 및 조기 공급 방안 마련 
사업 여건 개선: 전매 제한 완화 및 규제 정상화, 조기 인허가 인센티브 및 절차 개선, 공사비 증액 
기준 마련 및 인력 확충, 분양 사업의 임대 사업 전환 촉진 
원활한 자금 지원: PF대출 보증 확대, 중도금 대출 지원 
비아파트 자금 조달 지원 및 규제 개선 
도심 공급 기반 확충: 정비사업 절차 및 소규모 사업 사업성 개선 
2024년 1월 
도심 공급 확대: 재건축·재개발 패스트트랙 도입 및 재건축 부담금 합리화, 1기 신도시 재정비사업의 
신속하고 내실 있는 계획 수립, 소규모 정비·도심 복합 사업 속도 개선 및 자금 지원 강화 
다양한 유형의 주택 공급 확대: 도시·건축 규제 완화 및 세제·금융 지원, 등록 임대 사업 여건 개선 
및 기업형 장기임대 활성화, 신축 매입 약정 확대, 전세 사기 예방 및 피해 지원 
신도시 등 공공주택 공급: 2024년 공공주택 14만 호 이상 공급, 신도시 조성 속도 제고 
건설 경기 활력 회복: PF대출 지원 및 유동성 지원, 공공 임대 참여 지분 조기 매각, 민간 사업장 
공공 인수, 재정 조기 집행 및 민자 사업 확대 
 
자료: 국토교통부 보도자료 요약 
 
■ 2024년 주택시장 3대 변수는 공급과 금리, 그리고 정책 
주택시장이 상승과 하락을 반복하면서 주택 경기 불확실성이 확대되고 있다. 특히 매수 수요 위축으
로 주택 매매 거래량이 급감하면서 향후 주택 경기에 대한 부정적 시각이 팽배하다. 무엇보다 여전히 
높은 금리가 부담으로 작용하고 있다. 주택 경기 불황기에 고금리로 인한 대출 부담은 주택 수요를 크
게 위축시킬 수밖에 없기 때문이다. 
 
그림Ⅰ-11. 매수우위지수 추이  표Ⅰ-2. 주요 금융기관 주택담보대출 금리(2024년 2월) 
 
 
상품

### PyMuPDFLoader 클래스로 pdf 파일 읽어들이기

In [31]:
%%time

"""
PyMuPDFLoader 클래스를 이용하여 pdf 파일 읽어들인 후
10번째 인덱스부터 5개 본문 내용 출력..
"""
file_path = "./data/2024_KB_부동산_보고서_최종.pdf"
pym = PyMuPDFLoader(file_path)

pages=pym.load_and_split()
pages

for doc in pages[10:15]:
    print(f"[본문내용] : \n{getRegular_Expression(doc.page_content)}")
    print

[본문내용] : 
5
2024 KB 부동산 보고서: 2024년 주택시장 진단과 전망
표Ⅰ-1. 공급 확대 정책 주요 내용
날짜
주요 내용
2023년 9월
공공 부문 공급물량 확대: 3기 신도시를 포함한 공급물량 확대 및 조기 공급 방안 마련
사업 여건 개선: 전매 제한 완화 및 규제 정상화, 조기 인허가 인센티브 및 절차 개선, 공사비 증액
기준 마련 및 인력 확충, 분양 사업의 임대 사업 전환 촉진
원활한 자금 지원: PF대출 보증 확대, 중도금 대출 지원
비아파트 자금 조달 지원 및 규제 개선
도심 공급 기반 확충: 정비사업 절차 및 소규모 사업 사업성 개선
2024년 1월
도심 공급 확대: 재건축·재개발 패스트트랙 도입 및 재건축 부담금 합리화, 1기 신도시 재정비사업의
신속하고 내실 있는 계획 수립, 소규모 정비·도심 복합 사업 속도 개선 및 자금 지원 강화
다양한 유형의 주택 공급 확대: 도시·건축 규제 완화 및 세제·금융 지원, 등록 임대 사업 여건 개선
및 기업형 장기임대 활성화, 신축 매입 약정 확대, 전세 사기 예방 및 피해 지원
신도시 등 공공주택 공급: 2024년 공공주택 14만 호 이상 공급, 신도시 조성 속도 제고
건설 경기 활력 회복: PF대출 지원 및 유동성 지원, 공공 임대 참여 지분 조기 매각, 민간 사업장
공공 인수, 재정 조기 집행 및 민자 사업 확대
자료: 국토교통부 보도자료 요약

■ 2024년 주택시장 3대 변수는 공급과 금리, 그리고 정책
주택시장이 상승과 하락을 반복하면서 주택 경기 불확실성이 확대되고 있다. 특히 매수 수요 위축으
로 주택 매매 거래량이 급감하면서 향후 주택 경기에 대한 부정적 시각이 팽배하다. 무엇보다 여전히
높은 금리가 부담으로 작용하고 있다. 주택 경기 불황기에 고금리로 인한 대출 부담은 주택 수요를 크
게 위축시킬 수밖에 없기 때문이다.

그림Ⅰ-11. 매수우위지수 추이
표Ⅰ-2. 주요 금융기관 주택담보대출 금리(2024년 2월)

상품명
당월
최저
당월
최고
전월
평균
KB주택담보대출
4.2

### PDFPlumberLoader 클래스를 이용하여 pdf 파일 읽어들이기

In [32]:
%%time

"""
PDFPlumberLoader 클래스를 이용하여 pdf 파일 읽어들인 후
10번째 인덱스부터 5개 본문 내용 출력..
"""

loader = PDFPlumberLoader(file_path)
pages = loader.load_and_split()
pages

for index in (10, 15, 1):
    print(f"[본문내용] : {getRegular_Expression(pages[index].page_content)}")
    print("="*100)

[본문내용] : 2024 KB 부동산 보고서: 2024년 주택시장 진단과 전망
표Ⅰ-1. 공급 확대 정책 주요 내용
날짜 주요 내용
공공 부문 공급물량 확대: 3기 신도시를 포함한 공급물량 확대 및 조기 공급 방안 마련
사업 여건 개선: 전매 제한 완화 및 규제 정상화, 조기 인허가 인센티브 및 절차 개선, 공사비 증액
기준 마련 및 인력 확충, 분양 사업의 임대 사업 전환 촉진
2023년 9월
원활한 자금 지원: PF대출 보증 확대, 중도금 대출 지원
비아파트 자금 조달 지원 및 규제 개선
도심 공급 기반 확충: 정비사업 절차 및 소규모 사업 사업성 개선
도심 공급 확대: 재건축·재개발 패스트트랙 도입 및 재건축 부담금 합리화, 1기 신도시 재정비사업의
신속하고 내실 있는 계획 수립, 소규모 정비·도심 복합 사업 속도 개선 및 자금 지원 강화
다양한 유형의 주택 공급 확대: 도시·건축 규제 완화 및 세제·금융 지원, 등록 임대 사업 여건 개선
2024년 1월 및 기업형 장기임대 활성화, 신축 매입 약정 확대, 전세 사기 예방 및 피해 지원
신도시 등 공공주택 공급: 2024년 공공주택 14만 호 이상 공급, 신도시 조성 속도 제고
건설 경기 활력 회복: PF대출 지원 및 유동성 지원, 공공 임대 참여 지분 조기 매각, 민간 사업장
공공 인수, 재정 조기 집행 및 민자 사업 확대
자료: 국토교통부 보도자료 요약
■ 2024년 주택시장 3대 변수는 공급과 금리, 그리고 정책
주택시장이 상승과 하락을 반복하면서 주택 경기 불확실성이 확대되고 있다. 특히 매수 수요 위축으
로 주택 매매 거래량이 급감하면서 향후 주택 경기에 대한 부정적 시각이 팽배하다. 무엇보다 여전히
높은 금리가 부담으로 작용하고 있다. 주택 경기 불황기에 고금리로 인한 대출 부담은 주택 수요를 크
게 위축시킬 수밖에 없기 때문이다.
그림Ⅰ-11. 매수우위지수 추이 표Ⅰ-2. 주요 금융기관 주택담보대출 금리(2024년 2월)
당월 당월 전월
상품명
최저 최고 평균
KB주택담보대출 4.24 5.14

### CSV 파일 불러들이기

In [33]:
"""
<CSV Loader>
 * CSV 파일은 행과 열로 이루어진 구조(`,`로 구분됨)로,
   - 랭체인에서는 이러한 구조의 CSV 파일을 효율적으로 로드하고
     처리할 수 있는 라이브러리를 제공하고 있음


 * CSV Loader 종류
   1. CSVLoader : 가장 기본적으로 사용되는 라이브러리
   2. UnstructuredCSVLoader : 좀 더 유연하게 처리가능한 라이브러리


<라이브러리 설치>
  * UnstructuredCSVLoader 라이브러리 내에서 unstructured를 사용하기에
    - 아래 라이브러리 설치 필요
      >uv pip install unstructured
"""


'\n<CSV Loader>\n * CSV 파일은 행과 열로 이루어진 구조(`,`로 구분됨)로,\n   - 랭체인에서는 이러한 구조의 CSV 파일을 효율적으로 로드하고\n     처리할 수 있는 라이브러리를 제공하고 있음\n\n\n * CSV Loader 종류\n   1. CSVLoader : 가장 기본적으로 사용되는 라이브러리\n   2. UnstructuredCSVLoader : 좀 더 유연하게 처리가능한 라이브러리\n\n\n<라이브러리 설치>\n  * UnstructuredCSVLoader 라이브러리 내에서 unstructured를 사용하기에\n    - 아래 라이브러리 설치 필요\n      >uv pip install unstructured\n'

#### 사용할 라이브러리 정의

In [34]:
from langchain_community.document_loaders import CSVLoader
from langchain_community.document_loaders import UnstructuredCSVLoader

### CSVLoader 클래스를 사용하여 csv 파일 읽어들이기

In [35]:
"""
<CSVLoader>
 - CSV 파일을 "행 단위"로 로드함
 - 각 행을 독립적인 문서(Document)로 변환하여 관리함
 - 이때 각 행의 독립적인 문서를 "청크"라고 칭함


 - CSV 파일의 특정 행 중에, 빈 행이 포함된 경우에도 하나의 청크(문서)로 처리됨
"""


'\n<CSVLoader>\n - CSV 파일을 "행 단위"로 로드함\n - 각 행을 독립적인 문서(Document)로 변환하여 관리함\n - 이때 각 행의 독립적인 문서를 "청크"라고 칭함\n\n\n - CSV 파일의 특정 행 중에, 빈 행이 포함된 경우에도 하나의 청크(문서)로 처리됨\n'

In [36]:
### CSV 파일 로드하기
file_path = "./data/서울시_부동산_실거래가_정보.csv"

# - CSVLoader 클래스 생성하기
loader = CSVLoader(file_path)
loader

# - csv 파일 읽어들이기(로드만 합니다.)
documents = loader.load()

print(f"행(청크) 수 : {len(documents)}")
documents

행(청크) 수 : 2001


[Document(metadata={'source': './data/서울시_부동산_실거래가_정보.csv', 'row': 0}, page_content='\ufeff접수연도: 2024\n자치구코드: 11440\n자치구명: 마포구\n법정동코드: 11100\n법정동명: 신수동\n지번구분: 1\n지번구분명: 대지\n본번: 228\n부번: 3\n건물명: 마인빌\n계약일: 20241031\n물건금액(만원): 37300\n건물면적(㎡): 32.48\n토지면적(㎡): 20\n층: 4\n권리구분: \n취소일: \n건축년도: 2022\n건물용도: 연립다세대\n신고구분: 직거래\n신고한 개업공인중개사 시군구명: '),
 Document(metadata={'source': './data/서울시_부동산_실거래가_정보.csv', 'row': 1}, page_content='\ufeff접수연도: 2024\n자치구코드: 11320\n자치구명: 도봉구\n법정동코드: 10800\n법정동명: 도봉동\n지번구분: 1\n지번구분명: 대지\n본번: 565\n부번: 9\n건물명: (565-9)\n계약일: 20241031\n물건금액(만원): 9900\n건물면적(㎡): 25.92\n토지면적(㎡): 21\n층: 4\n권리구분: \n취소일: \n건축년도: 2003\n건물용도: 연립다세대\n신고구분: 직거래\n신고한 개업공인중개사 시군구명: '),
 Document(metadata={'source': './data/서울시_부동산_실거래가_정보.csv', 'row': 2}, page_content='\ufeff접수연도: 2024\n자치구코드: 11500\n자치구명: 강서구\n법정동코드: 10900\n법정동명: 방화동\n지번구분: 1\n지번구분명: 대지\n본번: 620\n부번: 214\n건물명: (620-214)\n계약일: 20241031\n물건금액(만원): 10000\n건물면적(㎡): 27.37\n토지면적(㎡): 40.37\n층: 7\n권리구분: \n취소일: \n건축년도: 2011\n건물용도: 오피스텔\n신고구

In [37]:
"""
읽어들인 정보에서 0번째 인덱스부터 5개만 본문 내용 출력해주세요...
"""
i = 0
for doc in documents[:5]:
    i += 1
    print(f"[본문내용{i}] : {getRegular_Expression(doc.page_content)}")
    print("="*100)

[본문내용1] : ﻿접수연도: 2024
자치구코드: 11440
자치구명: 마포구
법정동코드: 11100
법정동명: 신수동
지번구분: 1
지번구분명: 대지
본번: 228
부번: 3
건물명: 마인빌
계약일: 20241031
물건금액(만원): 37300
건물면적(㎡): 32.48
토지면적(㎡): 20
층: 4
권리구분:
취소일:
건축년도: 2022
건물용도: 연립다세대
신고구분: 직거래
신고한 개업공인중개사 시군구명:
[본문내용2] : ﻿접수연도: 2024
자치구코드: 11320
자치구명: 도봉구
법정동코드: 10800
법정동명: 도봉동
지번구분: 1
지번구분명: 대지
본번: 565
부번: 9
건물명: (565-9)
계약일: 20241031
물건금액(만원): 9900
건물면적(㎡): 25.92
토지면적(㎡): 21
층: 4
권리구분:
취소일:
건축년도: 2003
건물용도: 연립다세대
신고구분: 직거래
신고한 개업공인중개사 시군구명:
[본문내용3] : ﻿접수연도: 2024
자치구코드: 11500
자치구명: 강서구
법정동코드: 10900
법정동명: 방화동
지번구분: 1
지번구분명: 대지
본번: 620
부번: 214
건물명: (620-214)
계약일: 20241031
물건금액(만원): 10000
건물면적(㎡): 27.37
토지면적(㎡): 40.37
층: 7
권리구분:
취소일:
건축년도: 2011
건물용도: 오피스텔
신고구분: 중개거래
신고한 개업공인중개사 시군구명: 서울 강서구
[본문내용4] : ﻿접수연도: 2024
자치구코드: 11680
자치구명: 강남구
법정동코드: 11200
법정동명: 자곡동
지번구분: 1
지번구분명: 대지
본번: 658
부번: 0
건물명: 강남힐스테이트에코
계약일: 20241031
물건금액(만원): 17000
건물면적(㎡): 24.118
토지면적(㎡): 31.983
층: 10
권리구분:
취소일:
건축년도: 2014
건물용도: 오피스텔
신고구분: 중개거래
신고한 개업공인중개사 시군구명: 서울 강남구
[본문내용5] : ﻿접수연도: 2024
자치구코드:

In [38]:
### 0번째 본문 내용을 딕셔너리 데이터로 변경해 주세요..
"""
{"접수연도":"2024", "자치구코드":"11440",...}
"""
doc = documents[0].page_content


In [39]:
print(doc)

접수연도: 2024
자치구코드: 11440
자치구명: 마포구
법정동코드: 11100
법정동명: 신수동
지번구분: 1
지번구분명: 대지
본번: 228
부번: 3
건물명: 마인빌
계약일: 20241031
물건금액(만원): 37300
건물면적(㎡): 32.48
토지면적(㎡): 20
층: 4
권리구분: 
취소일: 
건축년도: 2022
건물용도: 연립다세대
신고구분: 직거래
신고한 개업공인중개사 시군구명: 


In [40]:
doc
### 불필요한 문자 없애기
doc2 = doc.replace("\ufeff","")

### \n을 기준으로 분리시키기 -> list 타입으로 만들어짐
doc3 = doc2.split("\n")


In [41]:
d = doc3[1].split(": ")
print(d[0])
print(d[1])

k = d[0]
v = d[1]
# {k : v}

### 깨끗한 딕셔너리 변수 정의
dict_data = {}

### 같은 key값이 있으면 수정, 없으면 추가됨
dict_data[k] = v

자치구코드
11440


In [42]:
dict = {}
for i in doc3:
    doc_split = i.split(": ")

    dict[doc_split[0]] = doc_split[1]

dict

{'접수연도': '2024',
 '자치구코드': '11440',
 '자치구명': '마포구',
 '법정동코드': '11100',
 '법정동명': '신수동',
 '지번구분': '1',
 '지번구분명': '대지',
 '본번': '228',
 '부번': '3',
 '건물명': '마인빌',
 '계약일': '20241031',
 '물건금액(만원)': '37300',
 '건물면적(㎡)': '32.48',
 '토지면적(㎡)': '20',
 '층': '4',
 '권리구분': '',
 '취소일': '',
 '건축년도': '2022',
 '건물용도': '연립다세대',
 '신고구분': '직거래',
 '신고한 개업공인중개사 시군구명': ''}

In [43]:
print(dict)

{'접수연도': '2024', '자치구코드': '11440', '자치구명': '마포구', '법정동코드': '11100', '법정동명': '신수동', '지번구분': '1', '지번구분명': '대지', '본번': '228', '부번': '3', '건물명': '마인빌', '계약일': '20241031', '물건금액(만원)': '37300', '건물면적(㎡)': '32.48', '토지면적(㎡)': '20', '층': '4', '권리구분': '', '취소일': '', '건축년도': '2022', '건물용도': '연립다세대', '신고구분': '직거래', '신고한 개업공인중개사 시군구명': ''}


In [44]:
import pandas as pd

In [45]:
documents

[Document(metadata={'source': './data/서울시_부동산_실거래가_정보.csv', 'row': 0}, page_content='\ufeff접수연도: 2024\n자치구코드: 11440\n자치구명: 마포구\n법정동코드: 11100\n법정동명: 신수동\n지번구분: 1\n지번구분명: 대지\n본번: 228\n부번: 3\n건물명: 마인빌\n계약일: 20241031\n물건금액(만원): 37300\n건물면적(㎡): 32.48\n토지면적(㎡): 20\n층: 4\n권리구분: \n취소일: \n건축년도: 2022\n건물용도: 연립다세대\n신고구분: 직거래\n신고한 개업공인중개사 시군구명: '),
 Document(metadata={'source': './data/서울시_부동산_실거래가_정보.csv', 'row': 1}, page_content='\ufeff접수연도: 2024\n자치구코드: 11320\n자치구명: 도봉구\n법정동코드: 10800\n법정동명: 도봉동\n지번구분: 1\n지번구분명: 대지\n본번: 565\n부번: 9\n건물명: (565-9)\n계약일: 20241031\n물건금액(만원): 9900\n건물면적(㎡): 25.92\n토지면적(㎡): 21\n층: 4\n권리구분: \n취소일: \n건축년도: 2003\n건물용도: 연립다세대\n신고구분: 직거래\n신고한 개업공인중개사 시군구명: '),
 Document(metadata={'source': './data/서울시_부동산_실거래가_정보.csv', 'row': 2}, page_content='\ufeff접수연도: 2024\n자치구코드: 11500\n자치구명: 강서구\n법정동코드: 10900\n법정동명: 방화동\n지번구분: 1\n지번구분명: 대지\n본번: 620\n부번: 214\n건물명: (620-214)\n계약일: 20241031\n물건금액(만원): 10000\n건물면적(㎡): 27.37\n토지면적(㎡): 40.37\n층: 7\n권리구분: \n취소일: \n건축년도: 2011\n건물용도: 오피스텔\n신고구

In [46]:
len(documents)

2001

In [60]:
### documents 2001개에 대한 본문 내용을 딕셔너리 데이터로 변경하고,
### documents 리스트 데이터 2001개에 대해서 데이터프레임으로 만들어주세요.(데이터프레임 변수명은 df)
"""
{"접수연도":"2024", "자치구코드":"11440",...}
"""
dict_list = []

for i in range(0, len(documents), 1):
    doc = documents[i].page_content

    ### 불필요한 문자 없애기
    doc2 = doc.replace("\ufeff","")

    ### \n을 기준으로 분리시키기 -> list 타입으로 만들어짐
    doc3 = doc2.split("\n")

    ### 각 컬럼명 추출하여 처리
    dict = {}
    for d in doc3:
        doc_split = d.split(": ")

        dict[doc_split[0]] = doc_split[1]
        
    dict_list.append(dict)

dict_list
### DataFrame 생성법
"""
df = pd.DataFrame([{"":"", "":"", ......}], {"":"", "":"", ......}, ....)

df = pd.DataFrame([["", ""...], ["", ""...]....], columns=["컬럼명1"])
"""

df = pd.DataFrame(dict_list)
df
df.info()
df

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2001 entries, 0 to 2000
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype 
---  ------            --------------  ----- 
 0   접수연도              2001 non-null   object
 1   자치구코드             2001 non-null   object
 2   자치구명              2001 non-null   object
 3   법정동코드             2001 non-null   object
 4   법정동명              2001 non-null   object
 5   지번구분              2001 non-null   object
 6   지번구분명             2001 non-null   object
 7   본번                2001 non-null   object
 8   부번                2001 non-null   object
 9   건물명               2001 non-null   object
 10  계약일               2001 non-null   object
 11  물건금액(만원)          2001 non-null   object
 12  건물면적(㎡)           2001 non-null   object
 13  토지면적(㎡)           2001 non-null   object
 14  층                 2001 non-null   object
 15  권리구분              2001 non-null   object
 16  취소일               2001 non-null   object
 17  건축년도          

,접수연도,자치구코드,자치구명,법정동코드,법정동명,지번구분,지번구분명,본번,부번,건물명,...,물건금액(만원),건물면적(㎡),토지면적(㎡),층,권리구분,취소일,건축년도,건물용도,신고구분,신고한 개업공인중개사 시군구명
0,2024,11440,마포구,11100,신수동,1,대지,228,3,마인빌,...,37300,32.48,20,4,,,2022,연립다세대,직거래,
1,2024,11320,도봉구,10800,도봉동,1,대지,565,9,(565-9),...,9900,25.92,21,4,,,2003,연립다세대,직거래,
2,2024,11500,강서구,10900,방화동,1,대지,620,214,(620-214),...,10000,27.37,40.37,7,,,2011,오피스텔,중개거래,서울 강서구
3,2024,11680,강남구,11200,자곡동,1,대지,658,0,강남힐스테이트에코,...,17000,24.118,31.983,10,,,2014,오피스텔,중개거래,서울 강남구
4,2024,11500,강서구,10300,화곡동,1,대지,343,44,계명아파트나동,...,39800,74.79,0,3,,,2001,아파트,중개거래,서울 강서구
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1996,2024,11545,금천구,10200,독산동,1,대지,234,36,성진빌라A동,...,20000,53.08,30,4,,,2003,연립다세대,중개거래,서울 금천구
1997,2024,11380,은평구,10900,신사동,1,대지,355,0,현대2,...,56000,84.83,0,14,,,1994,아파트,중개거래,서울 은평구
1998,2024,11590,동작구,10700,사당동,1,대지,236,2,샤인빌,...,46000,64.43,35,5,,,2005,연립다세대,중개거래,서울 동작구
1999,2024,11620,관악구,10100,봉천동,1,대지,1712,0,관악드림(삼성),...,95000,84.96,0,10,,,2003,아파트,중개거래,서울 관악구


In [54]:
# df["접수연도"].astype('int')
df["접수연도"].unique()


array(['2024', ''], dtype=object)

In [62]:
df1=[df["접수연도"].replace('', '2024')]
df1

[0       2024
 1       2024
 2       2024
 3       2024
 4       2024
         ... 
 1996    2024
 1997    2024
 1998    2024
 1999    2024
 2000    2024
 Name: 접수연도, Length: 2001, dtype: object]

TypeError: list indices must be integers or slices, not str

In [50]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2001 entries, 0 to 2000
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype 
---  ------            --------------  ----- 
 0   접수연도              2001 non-null   object
 1   자치구코드             2001 non-null   object
 2   자치구명              2001 non-null   object
 3   법정동코드             2001 non-null   object
 4   법정동명              2001 non-null   object
 5   지번구분              2001 non-null   object
 6   지번구분명             2001 non-null   object
 7   본번                2001 non-null   object
 8   부번                2001 non-null   object
 9   건물명               2001 non-null   object
 10  계약일               2001 non-null   object
 11  물건금액(만원)          2001 non-null   object
 12  건물면적(㎡)           2001 non-null   object
 13  토지면적(㎡)           2001 non-null   object
 14  층                 2001 non-null   object
 15  권리구분              2001 non-null   object
 16  취소일               2001 non-null   object
 17  건축년도          

### UnstructuredCSVLoader 클래스 사용하기

In [ ]:
"""
<UnstructuredCSVLoader 라이브러리>
 * CSV 파일을 유연하게 처리할 수 있는 라이브러리
 * 'single'모드와 'elements' 모드를 통해 다양한 방식으로 로드가능
   - single 모드 : CSV 파일 전체를 하나의 문서(청크)로 로드하는 기능(Unstructured)
   - elements 모드 : 요소(element)별로 분리하여 각 요소를 문서(청크)로 로드하는 기능
                  : 각 행을 독립적인 요소로 분리하여 문서(청크) 처리함
                  : elements 모드의 경우에는
                    => 표 형태를 갖출 수 있도록 HTML의 Table 태그형태로 제공하는 기능을 가지고 있음
                  : 로드한 데이터양이 많은 경우에는 주로 elements 모드를 사용함
* 작성 문법(클래스 생성 문법) : UnstructuredCSVLoader(파일위치, mode="elements")
"""

In [70]:
"""
* Data 폴더의 csv 파일을 UnstructuredCSVLoader 클래스를 이용하여 읽어들이기
  - 읽어들인 후 청크(문서)의 갯수 출력해 주세요
"""

file_path = "./data/서울시_부동산_실거래가_정보.csv"

loader = UnstructuredCSVLoader(file_path, mode="elements")

documents = loader.load()

print(len(documents))
documents

1


[Document(metadata={'source': './data/서울시_부동산_실거래가_정보.csv', 'file_directory': './data', 'filename': '서울시_부동산_실거래가_정보.csv', 'last_modified': '2026-11-05T14:48:02', 'text_as_html': "<table><tr><td>접수연도</td><td>자치구코드</td><td>자치구명</td><td>법정동코드</td><td>법정동명</td><td>지번구분</td><td>지번구분명</td><td>본번</td><td>부번</td><td>건물명</td><td>계약일</td><td>물건금액(만원)</td><td>건물면적(㎡)</td><td>토지면적(㎡)</td><td>층</td><td>권리구분</td><td>취소일</td><td>건축년도</td><td>건물용도</td><td>신고구분</td><td>신고한 개업공인중개사 시군구명</td></tr><tr><td>2024</td><td>11440</td><td>마포구</td><td>11100</td><td>신수동</td><td>1</td><td>대지</td><td>228</td><td>3</td><td>마인빌</td><td>20241031</td><td>37300</td><td>32.48</td><td>20</td><td>4</td><td/><td/><td>2022</td><td>연립다세대</td><td>직거래</td><td/></tr><tr><td>2024</td><td>11320</td><td>도봉구</td><td>10800</td><td>도봉동</td><td>1</td><td>대지</td><td>565</td><td>9</td><td>(565-9)</td><td>20241031</td><td>9900</td><td>25.92</td><td>21</td><td>4</td><td/><td/><td>2003</td><td>연립다세대</td><td>직거래</td><td/></tr><tr><td>2024</t

In [85]:
### HTML의 표(Table)로 제공된 데이터 추출하기 (text_as_html 속성의 값 추출)
doc = documents[0]
str_html = doc.metadata['text_as_html']
str_html

"<table><tr><td>접수연도</td><td>자치구코드</td><td>자치구명</td><td>법정동코드</td><td>법정동명</td><td>지번구분</td><td>지번구분명</td><td>본번</td><td>부번</td><td>건물명</td><td>계약일</td><td>물건금액(만원)</td><td>건물면적(㎡)</td><td>토지면적(㎡)</td><td>층</td><td>권리구분</td><td>취소일</td><td>건축년도</td><td>건물용도</td><td>신고구분</td><td>신고한 개업공인중개사 시군구명</td></tr><tr><td>2024</td><td>11440</td><td>마포구</td><td>11100</td><td>신수동</td><td>1</td><td>대지</td><td>228</td><td>3</td><td>마인빌</td><td>20241031</td><td>37300</td><td>32.48</td><td>20</td><td>4</td><td/><td/><td>2022</td><td>연립다세대</td><td>직거래</td><td/></tr><tr><td>2024</td><td>11320</td><td>도봉구</td><td>10800</td><td>도봉동</td><td>1</td><td>대지</td><td>565</td><td>9</td><td>(565-9)</td><td>20241031</td><td>9900</td><td>25.92</td><td>21</td><td>4</td><td/><td/><td>2003</td><td>연립다세대</td><td>직거래</td><td/></tr><tr><td>2024</td><td>11500</td><td>강서구</td><td>10900</td><td>방화동</td><td>1</td><td>대지</td><td>620</td><td>214</td><td>(620-214)</td><td>20241031</td><td>10000</td><td>27.37</td><td>40.37</td><td

In [95]:
### 문자열 html에 의미 부여시키기

### 문자열 HTML 데이터를 실제 HTML 파일 객체처럼 의미부여시키기
from io import StringIO

### *.html 파일과 같은 의미로 만듦
html_object = StringIO(str_html)
html_object

### 판다스(pandas)의 read_html(파일, 컬럼제목 위치)
df = pd.read_html(html_object, header=0)[0]
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2001 entries, 0 to 2000
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   접수연도              1999 non-null   float64
 1   자치구코드             2000 non-null   float64
 2   자치구명              2000 non-null   object 
 3   법정동코드             2000 non-null   float64
 4   법정동명              2000 non-null   object 
 5   지번구분              1937 non-null   float64
 6   지번구분명             1937 non-null   object 
 7   본번                1937 non-null   float64
 8   부번                1937 non-null   float64
 9   건물명               1937 non-null   object 
 10  계약일               2000 non-null   float64
 11  물건금액(만원)          2000 non-null   float64
 12  건물면적(㎡)           2000 non-null   float64
 13  토지면적(㎡)           2000 non-null   float64
 14  층                 1937 non-null   float64
 15  권리구분              27 non-null     object 
 16  취소일               30 non-null     float64


In [ ]:
# - 문자열 구조
str_html

"<table><tr><td>접수연도</td><td>자치구코드</td><td>자치구명</td><td>법정동코드</td><td>법정동명</td><td>지번구분</td><td>지번구분명</td><td>본번</td><td>부번</td><td>건물명</td><td>계약일</td><td>물건금액(만원)</td><td>건물면적(㎡)</td><td>토지면적(㎡)</td><td>층</td><td>권리구분</td><td>취소일</td><td>건축년도</td><td>건물용도</td><td>신고구분</td><td>신고한 개업공인중개사 시군구명</td></tr><tr><td>2024</td><td>11440</td><td>마포구</td><td>11100</td><td>신수동</td><td>1</td><td>대지</td><td>228</td><td>3</td><td>마인빌</td><td>20241031</td><td>37300</td><td>32.48</td><td>20</td><td>4</td><td/><td/><td>2022</td><td>연립다세대</td><td>직거래</td><td/></tr><tr><td>2024</td><td>11320</td><td>도봉구</td><td>10800</td><td>도봉동</td><td>1</td><td>대지</td><td>565</td><td>9</td><td>(565-9)</td><td>20241031</td><td>9900</td><td>25.92</td><td>21</td><td>4</td><td/><td/><td>2003</td><td>연립다세대</td><td>직거래</td><td/></tr><tr><td>2024</td><td>11500</td><td>강서구</td><td>10900</td><td>방화동</td><td>1</td><td>대지</td><td>620</td><td>214</td><td>(620-214)</td><td>20241031</td><td>10000</td><td>27.37</td><td>40.37</td><td

In [ ]:
### 문자열 html 데이터를 실제 html 포멧으로 의미부여 시키는 라이브러리
from bs4 import BeautifulSoup

### html.parser : 문자열 내의 태그들을 html로 의미부여 시키는 파서(변환기, 모든 브라우저에 있음)
html = BeautifulSoup(str_html, "html.parser")

# - html 구조
html

<table><tr><td>접수연도</td><td>자치구코드</td><td>자치구명</td><td>법정동코드</td><td>법정동명</td><td>지번구분</td><td>지번구분명</td><td>본번</td><td>부번</td><td>건물명</td><td>계약일</td><td>물건금액(만원)</td><td>건물면적(㎡)</td><td>토지면적(㎡)</td><td>층</td><td>권리구분</td><td>취소일</td><td>건축년도</td><td>건물용도</td><td>신고구분</td><td>신고한 개업공인중개사 시군구명</td></tr><tr><td>2024</td><td>11440</td><td>마포구</td><td>11100</td><td>신수동</td><td>1</td><td>대지</td><td>228</td><td>3</td><td>마인빌</td><td>20241031</td><td>37300</td><td>32.48</td><td>20</td><td>4</td><td></td><td></td><td>2022</td><td>연립다세대</td><td>직거래</td><td></td></tr><tr><td>2024</td><td>11320</td><td>도봉구</td><td>10800</td><td>도봉동</td><td>1</td><td>대지</td><td>565</td><td>9</td><td>(565-9)</td><td>20241031</td><td>9900</td><td>25.92</td><td>21</td><td>4</td><td></td><td></td><td>2003</td><td>연립다세대</td><td>직거래</td><td></td></tr><tr><td>2024</td><td>11500</td><td>강서구</td><td>10900</td><td>방화동</td><td>1</td><td>대지</td><td>620</td><td>214</td><td>(620-214)</td><td>20241031</td><td>10000</td><td>27.3